In [1]:
!pip install -q gradio

In [2]:
# ==========================================================
# IMPORTACIÓN DE LIBRERÍAS
# ==========================================================

import os
import json
import zipfile

import numpy as np
import pandas as pd
import tensorflow as tf
import gradio as gr

from PIL import Image
from google.colab import files

print("TensorFlow:", tf.__version__)
print("Gradio:", gr.__version__)

TensorFlow: 2.20.0
Gradio: 6.19.0


In [3]:
# ==========================================================
# CONECTAR GOOGLE DRIVE
# ==========================================================

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# ==========================================================
# RUTA DONDE ESTÁN LOS MODELOS
# ==========================================================

RUTA = "/content/drive/MyDrive/Proyecto_IA_4_CNN/modelos"

import os

print("Archivos encontrados:\n")

for archivo in os.listdir(RUTA):
    print(archivo)

Archivos encontrados:

mejor_vgg16_final.keras
mejor_mobilenetv2_fase1.keras
mejor_resnet50_final.keras
class_names.json
mejor_efficientnetb0_fase1.keras


In [5]:
# ==========================================================
# FUNCIONES DE PREPROCESAMIENTO
# ==========================================================

from tensorflow.keras.applications.vgg16 import (
    preprocess_input as preprocess_vgg16
)

from tensorflow.keras.applications.resnet50 import (
    preprocess_input as preprocess_resnet50
)

from tensorflow.keras.applications.mobilenet_v2 import (
    preprocess_input as preprocess_mobilenetv2
)

from tensorflow.keras.applications.efficientnet import (
    preprocess_input as preprocess_efficientnet
)

In [7]:
# ==========================================================
# FUNCIÓN PARA CARGAR MODELOS CON CAPA LAMBDA
# ==========================================================

def cargar_modelo_seguro(ruta, funcion_preprocesamiento):
    return tf.keras.models.load_model(
        ruta,
        custom_objects={
            "preprocess_input": funcion_preprocesamiento
        },
        compile=False,
        safe_mode=False
    )

In [8]:
# ==========================================================
# CARGAR LOS CUATRO MODELOS ENTRENADOS
# DESDE GOOGLE DRIVE
# ==========================================================

RUTA = "/content/drive/MyDrive/Proyecto_IA_4_CNN/modelos"

modelos = {
    "VGG16": cargar_modelo_seguro(
        f"{RUTA}/mejor_vgg16_final.keras",
        preprocess_vgg16
    ),

    "ResNet50": cargar_modelo_seguro(
        f"{RUTA}/mejor_resnet50_final.keras",
        preprocess_resnet50
    ),

    "MobileNetV2": cargar_modelo_seguro(
        f"{RUTA}/mejor_mobilenetv2_fase1.keras",
        preprocess_mobilenetv2
    ),

    "EfficientNetB0": cargar_modelo_seguro(
        f"{RUTA}/mejor_efficientnetb0_fase1.keras",
        preprocess_efficientnet
    )
}

with open(
    f"{RUTA}/class_names.json",
    "r",
    encoding="utf-8"
) as archivo:
    CLASS_NAMES = json.load(archivo)

print("\n" + "="*70)
print("SISTEMA DE DETECCIÓN DE DEFECTOS ESTRUCTURALES")
print("="*70)

for nombre in modelos:
    print(f"✓ {nombre} cargado correctamente")

print("\nClases disponibles:")
print(CLASS_NAMES)

print("\nTodos los modelos fueron cargados correctamente.")
print("="*70)


SISTEMA DE DETECCIÓN DE DEFECTOS ESTRUCTURALES
✓ VGG16 cargado correctamente
✓ ResNet50 cargado correctamente
✓ MobileNetV2 cargado correctamente
✓ EfficientNetB0 cargado correctamente

Clases disponibles:
['algae', 'major_crack', 'minor_crack', 'normal', 'peeling', 'spalling', 'stain']

Todos los modelos fueron cargados correctamente.


In [9]:
# ==========================================================
# FUNCIÓN DE PREDICCIÓN COMPARATIVA
# ==========================================================

def analizar_imagen(imagen, clase_real="No especificada"):

    if imagen is None:
        raise gr.Error("Primero debes subir una imagen.")

    # Asegurar formato RGB
    imagen_pil = Image.fromarray(imagen).convert("RGB")

    # Mismo tamaño utilizado durante el entrenamiento
    imagen_redimensionada = imagen_pil.resize((224, 224))

    # Convertir a matriz numérica
    imagen_array = np.asarray(
        imagen_redimensionada,
        dtype=np.float32
    )

    # Forma final: (1, 224, 224, 3)
    lote = np.expand_dims(imagen_array, axis=0)

    filas = []
    probabilidades_modelos = []
    predicciones_modelos = []

    for nombre_modelo, modelo in modelos.items():

        probabilidades = modelo.predict(
            lote,
            verbose=0
        )[0]

        indice = int(np.argmax(probabilidades))
        clase_predicha = CLASS_NAMES[indice]
        confianza = float(probabilidades[indice] * 100)

        probabilidades_modelos.append(probabilidades)
        predicciones_modelos.append(clase_predicha)

        if clase_real != "No especificada":
            estado = (
                "Correcto"
                if clase_predicha == clase_real
                else "Incorrecto"
            )
        else:
            estado = "Sin verificar"

        filas.append({
            "Modelo": nombre_modelo,
            "Predicción": clase_predicha,
            "Confianza (%)": round(confianza, 2),
            "Resultado": estado
        })

    # ======================================================
    # ENSAMBLE: PROMEDIO DE LAS PROBABILIDADES
    # ======================================================

    promedio_probabilidades = np.mean(
        np.array(probabilidades_modelos),
        axis=0
    )

    indice_ensemble = int(
        np.argmax(promedio_probabilidades)
    )

    clase_ensemble = CLASS_NAMES[indice_ensemble]

    confianza_ensemble = float(
        promedio_probabilidades[indice_ensemble] * 100
    )

    # Modelo individual con mayor confianza
    mejor_fila = max(
        filas,
        key=lambda fila: fila["Confianza (%)"]
    )

    # Consenso por mayoría
    valores, cantidades = np.unique(
        predicciones_modelos,
        return_counts=True
    )

    indice_mayoria = int(np.argmax(cantidades))
    clase_consenso = valores[indice_mayoria]
    votos_consenso = int(cantidades[indice_mayoria])

    # Probabilidades del ensamble para gr.Label
    probabilidades_ensemble = {
        clase: float(probabilidad)
        for clase, probabilidad in zip(
            CLASS_NAMES,
            promedio_probabilidades
        )
    }

    resumen = f"""
## Resultado del análisis

### Predicción conjunta
**{clase_ensemble}**

**Confianza del ensamble:** {confianza_ensemble:.2f} %

### Consenso entre modelos
**{clase_consenso}** recibió **{votos_consenso} de 4 votos**.

### Modelo con mayor confianza
**{mejor_fila["Modelo"]}**

Predijo **{mejor_fila["Predicción"]}** con una confianza de
**{mejor_fila["Confianza (%)"]:.2f} %**.
"""

    if clase_real != "No especificada":
        modelos_correctos = [
            fila["Modelo"]
            for fila in filas
            if fila["Resultado"] == "Correcto"
        ]

        if modelos_correctos:
            resumen += (
                "\n\n### Modelos que acertaron la clase real\n"
                + ", ".join(modelos_correctos)
            )
        else:
            resumen += (
                "\n\n### Verificación\n"
                "Ningún modelo coincidió con la clase real indicada."
            )

    tabla = pd.DataFrame(filas)

    return (
        tabla,
        probabilidades_ensemble,
        resumen
    )

In [10]:
# ==========================================================
# INTERFAZ GRÁFICA COMPARATIVA
# ==========================================================

with gr.Blocks(
    title="Comparador de CNN para defectos estructurales"
) as app:

    gr.Markdown(
        """
        # Sistema inteligente de inspección estructural

        Sube una fotografía de una pared o superficie de concreto.
        La imagen será analizada por **VGG16, ResNet50,
        MobileNetV2 y EfficientNetB0**.

        El sistema presentará las predicciones individuales,
        el consenso y una predicción conjunta.
        """
    )

    with gr.Row():

        with gr.Column():

            imagen_entrada = gr.Image(
                type="numpy",
                label="Imagen de la superficie"
            )

            clase_real = gr.Dropdown(
                choices=["No especificada"] + CLASS_NAMES,
                value="No especificada",
                label="Clase real, si se conoce"
            )

            boton_analizar = gr.Button(
                "Analizar con las 4 CNN",
                variant="primary"
            )

            boton_limpiar = gr.ClearButton(
                components=[
                    imagen_entrada,
                    clase_real
                ],
                value="Limpiar"
            )

        with gr.Column():

            resumen_salida = gr.Markdown(
                value="Sube una imagen para comenzar."
            )

            ensemble_salida = gr.Label(
                num_top_classes=7,
                label="Probabilidades de la predicción conjunta"
            )

    tabla_salida = gr.Dataframe(
        headers=[
            "Modelo",
            "Predicción",
            "Confianza (%)",
            "Resultado"
        ],
        datatype=[
            "str",
            "str",
            "number",
            "str"
        ],
        interactive=False,
        label="Comparación entre arquitecturas"
    )

    boton_analizar.click(
        fn=analizar_imagen,
        inputs=[
            imagen_entrada,
            clase_real
        ],
        outputs=[
            tabla_salida,
            ensemble_salida,
            resumen_salida
        ]
    )

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://81c85082dec0d70bfe.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
